# model 

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
API_key = os.getenv("GOOGLE_API_KEY")
llm_model = "gemini-2.5-flash"

In [4]:
os.environ["LANGCHAIN_PROJECT"] = "memory schema collection"

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [6]:
model = ChatGoogleGenerativeAI(
                    model=llm_model,
                    temperature=0,
                    timeout=None,
                    max_retries=2)

# Goals 

- Sometimes we want to save memories to a collection rather than single profile.
- We'll also show how to use Trustcall to update this collection.    

# Defining a collection schema
- We can define a collection schema as a Pydantic object.

In [7]:
from pydantic import BaseModel, Field

class Memory(BaseModel):
    content: str = Field(description="The main content of the memory. For example: User expressed interest in learning about French.")

class MemoryCollection(BaseModel):
    memories: list[Memory] = Field(description="A list of memories about the user.")

In [8]:
from langchain_core.messages import HumanMessage

In [9]:
# Bind schema to model
model_with_structure = model.with_structured_output(MemoryCollection)

In [12]:
# Invoke the model to produce structured output that matches the schema
memory_collection = model_with_structure.invoke([HumanMessage("My name is Lance. I like to bike.")])

In [13]:
memory_collection.memories

[Memory(content="User's name is Lance."),
 Memory(content='User likes to bike.')]

In [15]:
memory_collection.memories[0].model_dump()

{'content': "User's name is Lance."}

In [16]:
memory_collection.memories[1].model_dump()

{'content': 'User likes to bike.'}

## Save dictionary representation of each memory to the store.

In [17]:
import uuid
from langgraph.store.memory import InMemoryStore

In [18]:
# Initialize the in-memory store
in_memory_store = InMemoryStore()

In [19]:
# Namespace for the memory to save
user_id = "1"
namespace_for_memory = (user_id, "memories")

In [21]:
key = str(uuid.uuid4())
value = memory_collection.memories[0].model_dump()

In [22]:
in_memory_store.put(namespace_for_memory, key, value)

In [26]:
key = str(uuid.uuid4())
value = memory_collection.memories[1].model_dump()
in_memory_store.put(namespace_for_memory, key, value)

## Search for memories in the store.

In [27]:
# Search 
for m in in_memory_store.search(namespace_for_memory):
    print(m.dict())

{'namespace': ['1', 'memories'], 'key': '435bea15-34a3-4f4e-b5f5-db51c514fff6', 'value': {'content': "User's name is Lance."}, 'created_at': '2026-04-19T15:39:43.639319+00:00', 'updated_at': '2026-04-19T15:39:43.639322+00:00', 'score': None}
{'namespace': ['1', 'memories'], 'key': 'e14dc3f6-67f2-489b-8c4c-40cd98e56407', 'value': {'content': 'User likes to bike.'}, 'created_at': '2026-04-19T15:40:04.005848+00:00', 'updated_at': '2026-04-19T15:40:04.005852+00:00', 'score': None}
